# SLM-INDO · Notebook A: data FineWeb-2 + tokenizer

**Pengaturan Kaggle:** Accelerator = *None* (CPU), Internet = *On*. Jalankan dengan **Save Version → Save & Run All** (latar belakang, maks. 12 jam).

Keluaran di `/kaggle/working`:
- `eval/` (FLORES, SmSA, Wikipedia uji, test FineWeb-2, indeks dekontaminasi)
- `clean/` (parquet bersih, untuk ablation vocab)
- `tok16k/` (SentencePiece BPE 16.384)
- `tokens16k/` (`train/val/test.bin` + `meta.json`), dipakai Notebook B
- `audit.md`

Lihat KONSEP.md §9 dan README.md.

In [ ]:
import subprocess, os
def sh(cmd):
    """Jalankan perintah shell, tampilkan output langsung, hentikan notebook bila gagal."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"perintah gagal: {cmd}")

In [ ]:
sh("nproc; free -g; df -h /kaggle/working /tmp")
sh("rm -rf /tmp/SLM-INDO && git clone -q https://github.com/rexxar280903/SLM-INDO.git /tmp/SLM-INDO")
sh("cd /tmp/SLM-INDO && git log --oneline -1 | tee /kaggle/working/code_commit.txt")
sh("pip install -q sentencepiece pyarrow huggingface_hub")
os.chdir("/tmp/SLM-INDO")

In [ ]:
# Langkah 0: set evaluasi + indeks dekontaminasi
sh("python scripts/data/00_eval_sets.py --out /kaggle/working/eval")

In [ ]:
# Langkah 2: audit sampel (seed berbeda dari audit laptop)
sh("python scripts/data/02_audit.py --row-groups 3 --seed 11 --out /kaggle/working/audit.md")

In [ ]:
# Langkah 1: sampel FineWeb-2 → korpus bersih (±12 GB teks train ≈ 2,7B token)
sh("python scripts/data/01_sample_fineweb2.py --eval-dir /kaggle/working/eval --out /kaggle/working/clean --target-gb 12 --workers $(nproc)")

In [ ]:
# Langkah 3: tokenizer BPE 16.384 dari 1 GB teks train
sh("python scripts/data/03_train_tokenizer.py --clean /kaggle/working/clean --out /kaggle/working/tok16k --vocab 16384")

In [ ]:
# Langkah 4: tokenisasi → uint16
sh("python scripts/data/04_tokenize.py --clean /kaggle/working/clean --eval-dir /kaggle/working/eval --tok /kaggle/working/tok16k --out /kaggle/working/tokens16k --workers $(nproc)")

In [ ]:
sh("du -sh /kaggle/working/*; cat /kaggle/working/tokens16k/meta.json | head -40; cat /kaggle/working/tok16k/tokenizer_stats.json")